# Fine-tune QWEN in Google Colab

This notebook guide provides a comprehensive overview of using the `transformers` Python package to efficiently train a custom model. It covers the following techniques:

1. Utilizing model, tokenizer, and dataset loading functionalities from Hugging Face.
2. Performing basic data cleaning.
3. Training the model with basic modeling techniques, including quantization, such as qlora in this instance.
4. Evaluating the model's performance on test set.
5. Saving your custom model and preparing it for deployment.

## Preliminary Preparation

Before proceeding with model training, ensure your environment is properly configured by following these steps:

1. Install the necessary Python packages.
2. Import the required libraries.

In [1]:
!pip install -q h5py typing-extensions wheel fschat
# !pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 fschat
# !pip install -q -U bitsandbytes
# !pip install -q -U git+https://github.com/huggingface/transformers.git
# !pip install -q -U git+https://github.com/huggingface/peft.git
# !pip install -q -U git+https://github.com/huggingface/accelerate.git
# !pip install -q datasets

In [1]:
!nvidia-smi

Fri Nov 21 13:49:53 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.86.10              Driver Version: 535.86.10    CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:66:00.0 Off |                    0 |
| N/A   27C    P0              72W / 700W |      2MiB / 81559MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

## Load Pre-trained model and tokenizer

First let's load the model we are going to use - phoenix-inst-chat-7b! Note that the model itself is around 7B in full precision

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


model_id = "/llmchat/daixunlian/class_project/natural_language_progress/Qwen3-4B-Instruct-2507"
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=use_nested_quant,
#     bnb_4bit_quant_type=bnb_4bit_quant_type,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, 
                                             torch_dtype=torch.bfloat16,
                                             device_map={"":0})

/llmchat/daixunlian/verl/verl_daixl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:42<00:00,  9.41it/s, Materializing param=model.embed_tokens.weight]                        


Then we have to apply some preprocessing to the model to prepare it for training. For that use the `prepare_model_for_kbit_training` method from PEFT.

In [4]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [5]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [6]:
from peft import LoraConfig, get_peft_model
# You can try differnt parameter-effient strategy for model trianing, for more info, please check https://github.com/huggingface/peft
config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)

In [7]:
from fastchat.conversation import get_conv_template
device = "cuda"
model.eval()

@torch.no_grad()
def generate(prompt):
    input_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors='pt').to(device)
    outputs = model.generate(input_ids, do_sample=False, max_new_tokens=1024)
    return tokenizer.decode(*outputs, skip_special_tokens=True)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
response = generate(text)
print("-"*80)
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--------------------------------------------------------------------------------
user
Give me a short introduction to large language model.
assistant
A large language model (LLM) is a type of artificial intelligence system trained on vast amounts of text data from the internet, books, and other sources. It learns to understand and generate human-like language by identifying patterns in the data. These models consist of billions of parameters—weights that allow the system to make predictions about the next word in a sentence. LLMs can perform a wide range of tasks, such as answering questions, writing stories, coding, summarizing text, and more. Examples include GPT-3, GPT-4, and Llama. While powerful, they operate based on statistical patterns and do not possess true understanding or consciousness.


## Data Preparation

Let's load a common dataset, english quotes, to fine tune our model on famous quotes.

In [8]:
from datasets import load_dataset

# data = load_dataset("Abirate/english_quotes")
dataset = load_dataset("FreedomIntelligence/Huatuo26M-Lite")
dataset = dataset['train'].map(lambda sample: {"conversations": [{"role": "human", "value": sample['question']}, {"role": "gpt", "value": sample['answer']}]}, batched=False)

In [9]:
from torch.utils.data import random_split

In [10]:
print(len(dataset))

177703


In [11]:
train_dataset_size = int(len(dataset) *0.8)
val_dataset_size = len(dataset) - train_dataset_size
train_dataset_size, val_dataset_size = int(train_dataset_size * 0.1), int(val_dataset_size * 0.1)
print(f"train_dataset_size: {train_dataset_size}, val_dataset_size: {val_dataset_size}")


train_dataset_size: 14216, val_dataset_size: 3554


In [12]:
train_dataset, val_dataset, _ = random_split(dataset, [train_dataset_size, val_dataset_size, len(dataset)-train_dataset_size-val_dataset_size])


### Customized Dataset
Create a specialized dataset class named "InstructionDataset" designed to handle our custom dataset.

In [13]:
import json, copy
import transformers
from typing import Dict, Sequence, List
from dataclasses import dataclass
from torch.utils.data import Dataset

IGNORE_INDEX = -100
DEFAULT_PAD_TOKEN = "<pad>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_UNK_TOKEN = "<unk>"
default_conversation = get_conv_template('phoenix')

class InstructDataset(Dataset):
    def __init__(self, data: Sequence, tokenizer: transformers.PreTrainedTokenizer) -> None:
        super().__init__()
        self.tokenizer = tokenizer
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        sources = self.data[index]
        if isinstance(index, int):
            sources = [sources]
        data_dict = preprocess([e['conversations'] for e in sources], self.tokenizer)
        if isinstance(index, int):
            data_dict = dict(input_ids=data_dict["input_ids"][0], labels=data_dict["labels"][0])
        return data_dict

def preprocess(
        sources: Sequence[str],
        tokenizer: transformers.PreTrainedTokenizer,
        max_length=1024
) -> Dict:
    # add end signal and concatenate together
    conversations = []
    intermediates = []
    for source in sources:
        header = f"{default_conversation.system_message}"
        conversation, intermediate = _add_speaker_and_signal(header, source)
        conversations.append(conversation)
        intermediates.append(intermediate)

    # tokenize conversations
    conversations_tokenized = _tokenize_fn(conversations, tokenizer)
    input_ids = conversations_tokenized["input_ids"]
    targets = copy.deepcopy(input_ids)

    # keep only machine responses as targets
    assert len(targets) == len(intermediates)
    for target, inters in zip(targets, intermediates):
        mask = torch.zeros_like(target, dtype=torch.bool)
        for inter in inters:
            tokenized = _tokenize_fn(inter, tokenizer)
            start_idx = tokenized["input_ids"][0].size(0) - 1
            end_idx = tokenized["input_ids"][1].size(0)
            mask[start_idx:end_idx] = True
        target[~mask] = IGNORE_INDEX

    input_ids = input_ids[:max_length]
    targets = targets[:max_length]
    return dict(input_ids=input_ids, labels=targets)

def _add_speaker_and_signal(header, source, get_conversation=True):
    BEGIN_SIGNAL = DEFAULT_BOS_TOKEN
    END_SIGNAL = DEFAULT_EOS_TOKEN
    conversation = header
    intermediate = []
    for sentence in source:
        from_str = sentence["role"]
        if from_str.lower() == "human":
            from_str = default_conversation.roles[0]
        elif from_str.lower() == "gpt":
            from_str = default_conversation.roles[1]
        else:
            from_str = 'unknown'
        # store the string w/o and w/ the response
        value = (from_str + ": " + BEGIN_SIGNAL + sentence["value"] + END_SIGNAL)
        if sentence["role"].lower() == "gpt":
            start = conversation + from_str + ": " + BEGIN_SIGNAL
            end = conversation + value
            intermediate.append([start, end])
        if get_conversation:
            conversation += value
    return conversation, intermediate

def _tokenize_fn(strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
    tokenized_list = [
        tokenizer(
            text,
            return_tensors="pt",
            padding="longest",
            max_length=tokenizer.model_max_length,
            truncation=True,
        ) for text in strings
    ]
    input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
    input_ids_lens = labels_lens = [
        tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item()
        for tokenized in tokenized_list
    ]
    return dict(
        input_ids=input_ids,
        labels=labels,
        input_ids_lens=input_ids_lens,
        labels_lens=labels_lens,
    )

@dataclass
class DataCollatorForSupervisedDataset(object):
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id)
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [14]:
train_dataset = InstructDataset(train_dataset, tokenizer)
val_dataset = InstructDataset(val_dataset, tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

In [15]:
sample_data = train_dataset[1]

print("=" * 80)
print("Debuging: ")
print(sample_data)
print("-" * 80)
print(f"input_ids:\n{tokenizer.decode(sample_data['input_ids'])}")
# Filter out IGNORE_INDEX before decoding labels
z = [token for token in sample_data['labels'] if token != IGNORE_INDEX]
print("-" * 80)
print(f"labels:\n{tokenizer.decode(z)}")
print("=" * 80)

Debuging: 
{'input_ids': tensor([    32,   6236,   1948,    264,  22208,   3738,    323,    458,  20443,
         11229,  17847,     13,    576,  17847,   6696,  10950,     11,  11682,
            11,    323,  47787,  11253,    311,    279,   3738,    594,   4755,
           382,  33975,     25,    366,     82,     29,  35946,     20,     15,
         92015,  34187,  45181,     17, 104081, 103934,   3837, 110362,  80158,
        100397,  34187,   3837, 101150,  26939,     18, 105447, 104411, 110362,
         13343, 104685,  42192,   3837, 104685,  99859,     17,  35727,  80158,
         99518, 101161,   3837, 102021,  99917,  11319,    522,     82,     29,
         71703,     25,    366,     82,     29, 100345, 101214,  53481,   3837,
         87026,     20,     15,  92015,  34187,   3837, 110362,  45181,     17,
        104081, 103934,  80158, 101197,  16530, 104466,   3837, 104685,  18830,
        104685,  42192,   3837, 104685,  99859,     17,  35727,  80158,  99518,
        101161,

## Training

### General Training Hyperparameters

In [ ]:
# Set training parameters
training_arguments = transformers.TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    save_steps=0,
    logging_steps=1,
    learning_rate=2e-7,
    weight_decay=0.001,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
model.train()
trainer = transformers.Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [18]:
trainer.train()

Step,Training Loss
1,1.905700
2,2.078900
3,2.299100
4,2.343100
5,2.334000
6,2.460800
7,2.468800
8,2.427800
9,2.410900
10,2.372800


TrainOutput(global_step=223, training_loss=2.761221848795767, metrics={'train_runtime': 1633.371, 'train_samples_per_second': 8.703, 'train_steps_per_second': 0.137, 'total_flos': 5.840956356609638e+16, 'train_loss': 2.761221848795767, 'epoch': 1.0})

Once the training is completed, we can evaluate our model and get its perplexity on the validation set like this:

In [19]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 15.55


## Save Trained LoRA

In [20]:
output_path = "lora"
trainer.save_model(output_path)

### Test the trained model

In [ ]:
from fastchat.conversation import get_conv_template
device = "cuda"
model.eval()
@torch.no_grad()
def generate(prompt):
    input_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors='pt').to(device)
    outputs = trainer.model.generate(input_ids, do_sample=False, max_new_tokens=1024)
    return tokenizer.decode(*outputs, skip_special_tokens=True)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
response = generate(text)
print("-"*80)
print(response)
### 这里的报错是因为没有一次性运行导致的而不是代码不通

NameError: name 'model' is not defined

# Clean GPU Memory

In [22]:
# Empty VRAM
del model
del trainer
import gc
gc.collect()
gc.collect()

0

## Load the trained model back and integrate the trained LoRA within.

In [5]:
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Re-define quantization parameters if needed, or use the ones defined earlier
bnb_4bit_quant_type = "nf4" # or "fp4"
use_nested_quant = True # or False

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=use_nested_quant,
#     bnb_4bit_quant_type=bnb_4bit_quant_type,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )
output_path = "lora"
model_id = "/llmchat/daixunlian/class_project/natural_language_progress/Qwen3-4B-Instruct-2507"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map={"":0})
model = PeftModel.from_pretrained(model, output_path)
model = model.merge_and_unload()
model.config.max_length = 512
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
# tokenizer.pad_token = tokenizer.unk_token # This line is not needed and can be removed

Loading weights: 100%|██████████| 398/398 [00:09<00:00, 39.97it/s, Materializing param=model.embed_tokens.weight]                      


## Answer generation

In [6]:
from tqdm import tqdm
@torch.no_grad()
def generate(query_list, return_answer: bool = False):
    def conv_format(query):
        conv = get_conv_template('phoenix')
        conv.append_message(conv.roles[0], query)
        conv.append_message(conv.roles[1], None)
        return conv.get_prompt()
    query_list = [conv_format(query) for query in query_list]
    input_ids = tokenizer(query_list, padding=True, truncation=True, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda")
    n_input, n_seq = input_ids.shape[0], input_ids.shape[-1]
    output_ids = []
    step = 1
    for index in tqdm(range(0, n_input, step)):
        outputs = model.generate(
            input_ids=input_ids[index: min(n_input, index+step)],
            do_sample=False,
            max_new_tokens=1024,
            # temperature=0.7,
            repetition_penalty=1.0,
        )
        output_ids += outputs
    responses = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    if return_answer:
        return [response[len(query):].strip() for query, response in zip(query_list, responses)]
    return responses

# test
print("\n".join(generate(["What's the weather like today?", "Who are you?"])))


  0%|          | 0/2 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
100%|██████████| 2/2 [00:30<00:00, 15.11s/it]

A chat between a curious human and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the human's questions.

Human: <s>What's the weather like today?</s>Assistant: <s>What's the weather like today?</s>  
I'm sorry, but I can't provide real-time information like today's weather. To find out the current weather, I recommend checking a reliable weather website or app such as Weather.com, AccuWeather, or your smartphone's built-in weather feature. Let me know if you'd like tips on interpreting weather reports or understanding weather forecasts! 😊🌤️🌧️
A chat between a curious human and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the human's questions.

Human: <s>Who are you?</s>Assistant: <s>Who are you?</s>  
I'm Qwen, a large-scale language model developed by Alibaba Cloud's Tongyi Lab. You can call me Qwen. I'm designed to assist with a wide range of tasks, such as answering questions,

## Evaluate a trained model on a given test dataset

In [8]:
import os
import json
# TODO: correctly put test data files into an accessible path
test_file = "zh_med.json"
assert os.path.exists(test_file), "Invalid test_file path"

with open(test_file, 'r', encoding='utf-8') as reader:
    test_data = json.load(reader)
print(test_data[0])

['什么是医学伦理学，它在医疗领域有何重要性？', '医学伦理学是研究医疗领域伦理问题的学科。它涉及研究医疗专业人员、患者和其他相关利益相关者之间的伦理关系，以及在医疗实践中出现的道德困境。\n\n医学伦理学在医疗领域具有以下重要性：\n\n1. 保护患者权益：医学伦理学关注患者的权益和尊严。它确保医疗决策是以患者的最大利益为出发点，并尊重患者的自主权和知情同意权。\n\n2. 促进医务人员职业道德：医学伦理学提供了医务人员在面对道德困境时的指导原则，帮助他们保持专业的道德标准和行为规范。\n\n3. 增加医疗决策的公正性：医学伦理学关注公正和公平的医疗分配原则。它确保资源在医疗领域的分配是公正和可持续的。\n\n4. 促进研究伦理：医学伦理学对医学研究进行伦理审查，确保研究参与者的权益和福利得到保护，并确保研究过程是符合伦理标准的。\n\n5. 保护医疗机构声誉：医学伦理学的遵循有助于确保医疗机构遵守伦理原则，保护其声誉和公众信任度。\n\n总之，医学伦理学在医疗领域的重要性在于维护患者权益、指导医务人员的职业道德行为、促进医疗决策公正和保护医学研究伦理。它为医疗行业提供了一个道德框架，确保医疗服务的质量和道德高于一切。']


In [9]:
model_answers = generate([data[0] for data in test_data], return_answer=True)

100%|██████████| 20/20 [08:19<00:00, 24.97s/it]


In [11]:
for data, answer in zip(test_data, model_answers):
    data.append(answer)

In [10]:
with open("saved_data.json", 'w', encoding='utf-8') as writer:
    json.dump(test_data, writer, indent=4, ensure_ascii=False)